# Silverwing-ML Training

**Setup:** Runtime → Change runtime type → T4 GPU

Run all cells in order. Everything is self-contained.

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/silverwing'
os.makedirs(DRIVE, exist_ok=True)
print('Drive mounted:', DRIVE)

In [ ]:
# Cell 2: Install dependencies
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q pyyaml numpy
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - CHANGE RUNTIME TO GPU!"}')

In [ ]:
# Cell 3: Create project structure
import os, json, hashlib

ROOT = '/content/Silverwing-ML'
os.makedirs(ROOT, exist_ok=True)
os.chdir(ROOT)

# Create directory structure
dirs = [
    'foundation/model', 'foundation/training', 'foundation/tokenizer',
    'foundation/corpus', 'foundation/sft', 'foundation/inference',
    'foundation/curriculum', 'foundation/reasoning', 'foundation/alignment',
    'foundation/evaluation', 'foundation/math_corpus',
    'intelligence/mathematics', 'intelligence/reasoning',
    'intelligence/engineering', 'intelligence/memory',
    'intelligence/planning', 'intelligence/tools',
    'benchmarks/data', 'benchmarks/math',
    'serving/api', 'serving/runtime',
    'scripts', 'configs', 'tests',
    'experiments/tokenizer', 'experiments/corpus',
    'experiments/sft', 'experiments/reasoning',
    'experiments/curriculum', 'experiments/checkpoints',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print(f'Created {len(dirs)} directories at {ROOT}')

In [ ]:
# Cell 4: Build model from scratch
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, json, os

# === Model Config ===
VOCAB = 16384
N_LAYER = 12
N_HEAD = 12
N_EMBD = 512
BLOCK = 512

class RoPE(nn.Module):
    def __init__(self, dim, max_len=2048):
        super().__init__()
        inv = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_len).float()
        freq = torch.outer(t, inv)
        self.register_buffer('cos', freq.cos())
        self.register_buffer('sin', freq.sin())
    def forward(self, x):
        n = x.shape[-2]
        cos, sin = self.cos[:n], self.sin[:n]
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1).flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = dim // n_head
        self.qkv = nn.Linear(dim, 3*dim)
        self.proj = nn.Linear(dim, dim)
        self.rope = RoPE(self.head_dim)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_head, self.head_dim)
        q, k, v = qkv.unbind(2)
        q, k, v = [a.transpose(1,2) for a in (q, k, v)]
        q, k = self.rope(q), self.rope(k)
        mask = torch.triu(torch.ones(T, T, device=x.device), 1).bool()
        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(mask, float('-inf')).softmax(-1)
        y = (att @ v).transpose(1,2).reshape(B, T, C)
        return self.proj(y)

class MLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, 4*dim)
        self.proj = nn.Linear(4*dim, dim)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, dim, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, n_head)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class SilverwingDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB, N_EMBD)
        self.drop = nn.Dropout(0.1)
        self.blocks = nn.ModuleList([Block(N_EMBD, N_HEAD) for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.head = nn.Linear(N_EMBD, VOCAB, bias=False)
        self.head.weight = self.tok_emb.weight
    def forward(self, idx, targets=None):
        x = self.drop(self.tok_emb(idx))
        for b in self.blocks:
            x = b(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB), targets.view(-1)) if targets is not None else None
        return logits, loss

model = SilverwingDecoder()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params ({n_params/1e6:.1f}M)')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# Cell 5: Build tokenizer + corpus + train
import torch, json, os, hashlib, time
from pathlib import Path

# === Build corpus from training data ===
corpus_text = """
The quick brown fox jumps over the lazy dog. A stitch in time saves nine.
To be or not to be, that is the question. All that glitters is not gold.
Knowledge is power. Practice makes perfect. Time flies when having fun.
The early bird catches the worm. Actions speak louder than words.
Every cloud has a silver lining. Rome was not built in a day.
What goes around comes around. The pen is mightier than the sword.
Beauty is in the eye of the beholder. Birds of a feather flock together.
A rolling stone gathers no moss. Better late than never.
Curiosity killed the cat. Do not judge a book by its cover.
An apple a day keeps the doctor away. Two wrongs do not make a right.
The grass is always greener on the other side. Where there is smoke there is fire.
Necessity is the mother of invention. Honesty is the best policy.
Look before you leap. Many hands make light work.
What is 2 + 2? The answer is 4. What is 3 * 3? The answer is 9.
What is 10 - 5? The answer is 5. What is 12 / 4? The answer is 3.
What is 7 + 8? The answer is 15. What is 6 * 7? The answer is 42.
What is 100 - 37? The answer is 63. What is 15 * 2? The answer is 30.
The sum of 45 and 55 is 100. The product of 8 and 9 is 72.
1 + 1 = 2. 2 + 2 = 4. 3 + 3 = 6. 4 + 4 = 8. 5 + 5 = 10.
2 * 3 = 6. 4 * 5 = 20. 6 * 7 = 42. 8 * 9 = 72. 10 * 10 = 100.
10 - 3 = 7. 20 - 8 = 12. 50 - 25 = 25. 100 - 1 = 99.
""" * 100  # repeat for more training data

# === Simple BPE tokenizer ===
import re
from collections import Counter

def build_simple_tokenizer(text, vocab_size=256):
    chars = list(sorted(set(text)))
    vocab = {c: i for i, c in enumerate(chars)}
    merges = []
    tokens = list(text)
    for i in range(vocab_size - len(vocab)):
        pairs = Counter()
        for j in range(len(tokens)-1):
            pairs[(tokens[j], tokens[j+1])] += 1
        if not pairs: break
        best = pairs.most_common(1)[0][0]
        new_id = len(vocab)
        vocab[''.join(best)] = new_id
        merges.append(best)
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens)-1 and (tokens[j], tokens[j+1]) == best:
                new_tokens.append(''.join(best))
                j += 2
            else:
                new_tokens.append(tokens[j])
                j += 1
        tokens = new_tokens
    return vocab, merges

print('Building tokenizer...')
vocab, merges = build_simple_tokenizer(corpus_text, vocab_size=VOCAB)
print(f'Vocab size: {len(vocab)}')

def encode(text, vocab):
    tokens = list(text)
    for merge in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == merge:
                new_tokens.append(''.join(merge))
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return [vocab.get(t, 0) for t in tokens]

# Save tokenizer
tok_dir = 'experiments/tokenizer'
json.dump({'vocab': vocab, 'merges': [list(m) for m in merges]}, open(f'{tok_dir}/config.json', 'w'))
json.dump(vocab, open(f'{tok_dir}/vocab.json', 'w'))
json.dump([list(m) for m in merges], open(f'{tok_dir}/merges.json', 'w'))
print(f'Tokenizer saved to {tok_dir}/')

# Encode corpus
token_ids = encode(corpus_text, vocab)
print(f'Corpus tokens: {len(token_ids):,}')

# Create blocks
blocks = []
for i in range(0, len(token_ids) - BLOCK, BLOCK):
    blocks.append(token_ids[i:i+BLOCK+1])
print(f'Training blocks: {len(blocks)}')

In [ ]:
# Cell 6: Train (pretrain + SFT combined)
import torch, time, os
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Training config
EPOCHS = 20
BATCH = 4
LR = 3e-4
STEPS_PER_EPOCH = max(1, len(blocks) // BATCH)
MAX_STEPS = min(EPOCHS * STEPS_PER_EPOCH, 5000)

# Data
data = torch.tensor(blocks[:BATCH * (len(blocks)//BATCH)], dtype=torch.long)
X, Y = data[:,:-1], data[:,1:]
ds = TensorDataset(X, Y)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True)

# Optimizer
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_STEPS)

print(f'Training: {MAX_STEPS} steps, batch={BATCH}, lr={LR}, device={device}')
print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')
print('='*50)

model.train()
step = 0
loss_acc = 0
t0 = time.time()
best_loss = float('inf')

while step < MAX_STEPS:
    for bx, by in dl:
        bx, by = bx.to(device), by.to(device)
        _, loss = model(bx, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        opt.zero_grad()
        scheduler.step()
        loss_acc += loss.item()
        step += 1
        if step % 100 == 0:
            avg = loss_acc / 100
            elapsed = time.time() - t0
            print(f'Step {step}/{MAX_STEPS} | loss={avg:.3f} | lr={scheduler.get_last_lr()[0]:.2e} | {elapsed:.0f}s')
            loss_acc = 0
            if avg < best_loss:
                best_loss = avg
                torch.save(model.state_dict(), f'{DRIVE}/best.pt')
        if step >= MAX_STEPS:
            break

# Save final
torch.save(model.state_dict(), f'{DRIVE}/final.pt')
print('='*50)
print(f'Done! Best loss: {best_loss:.3f}')
print(f'Checkpoints saved to: {DRIVE}')

# List files
for f in os.listdir(DRIVE):
    if f.endswith('.pt'):
        mb = os.path.getsize(os.path.join(DRIVE, f)) / 1e6
        print(f'  {f} ({mb:.0f} MB)')

In [ ]:
# Cell 7: Test generation
import torch

model.eval()
model.to(device)

prompts = [
    'What is 2 + 2? ',
    'The answer to 3 * 3 is ',
    'Knowledge is ',
]

for prompt in prompts:
    tokens = encode(prompt, vocab)
    x = torch.tensor([tokens], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(32):
            logits, _ = model(x)
            next_tok = logits[:, -1, :].argmax(-1, keepdim=True)
            x = torch.cat([x, next_tok], dim=1)
    gen = x[0].tolist()
    # Decode
    inv_vocab = {v: k for k, v in vocab.items()}
    text = ''.join(inv_vocab.get(t, '?') for t in gen)
    print(f'Prompt: {prompt!r}')
    print(f'Output: {text!r}')
    print()